# NLP分析实验: 实体识别/关键词提取/摘要生成

> **对应模块**: `src/nlp_pipeline.py` (NLP分析层)
> **前置条件**: `pip install transformers keybert wordcloud torch`

## Part 1: 什么是NLP？

**NLP (自然语言处理)** 是让计算机理解人类语言的AI分支。

在智析项目中，NLP用于:
1. **命名实体识别 (NER)**: 找出文本中的人名、组织、地点
2. **关键词提取**: 用KeyBERT基于语义找到核心词汇
3. **自动摘要**: 用BART模型生成文档摘要
4. **词云**: 可视化展示文本中的高频词汇

In [ ]:
import sys
sys.path.insert(0, '..')
from src.nlp_pipeline import NLPPipeline, Entity, NLPResult, analyze_text

# 测试文本 (英文效果最好，因为默认模型是英文的)
sample_text = """
Artificial intelligence (AI) is transforming industries worldwide. 
Companies like Google, Microsoft, and OpenAI are leading the development 
of large language models. In 2024, the global AI market was valued at 
over $200 billion. Researchers at Stanford University and MIT have 
published groundbreaking papers on transformer architectures and 
reinforcement learning from human feedback (RLHF). The technology 
has applications in healthcare, finance, education, and autonomous 
vehicles. Sundar Pichai, CEO of Google, announced a $2 billion 
investment in AI research.
"""

print("=== 测试文本 ===")
print(sample_text[:200] + "...")

In [ ]:
# ========================================
# 1.1 关键词提取 (KeyBERT) — 最先运行，不需要大模型
# ========================================

nlp = NLPPipeline()

print("[1] 关键词提取...")
keywords = nlp.extract_keywords(sample_text, top_k=10)

print("\n提取到的关键词 (按相关度排序):")
for kw, score in keywords:
    bar = '█' * int(score * 30)
    print(f"  {kw:>30s}  {score:.4f}  {bar}")

In [ ]:
# ========================================
# 1.2 命名实体识别 (NER)
# ========================================
# 注意: 首次运行需要下载模型 (~400MB)

print("[2] 命名实体识别...")
entities = nlp.extract_entities(sample_text)

print(f"\n识别到 {len(entities)} 个实体:")

# 按类型分组
from collections import defaultdict
by_type = defaultdict(list)
for e in entities:
    by_type[e.label].append(e.text)

type_names = {'PER': '人名', 'ORG': '组织', 'LOC': '地点', 'MISC': '其他'}
for label, names in by_type.items():
    label_cn = type_names.get(label, label)
    unique_names = list(set(names))
    print(f"\n  {label_cn} ({label}): {', '.join(unique_names)}")

In [ ]:
# ========================================
# 1.3 自动摘要生成 (BART)
# ========================================
# 注意: 首次运行需要下载模型 (~1.5GB)
# 如果内存不够可以跳过这步

print("[3] 自动摘要生成...")
try:
    summary = nlp.generate_summary(sample_text)
    print(f"\n原文长度: {len(sample_text)} 字符")
    print(f"摘要长度: {len(summary)} 字符")
    print(f"\n摘要内容:\n{summary}")
except Exception as e:
    print(f"摘要生成跳过: {e}")
    print("(可能原因: 内存不足或模型下载失败)")

In [ ]:
# ========================================
# 1.4 词云可视化
# ========================================
import os
os.makedirs('../data/processed', exist_ok=True)

print("[4] 词云生成...")
path = nlp.generate_wordcloud(sample_text, '../data/processed/wordcloud.png')
if path:
    from PIL import Image
    import matplotlib.pyplot as plt
    
    img = Image.open(path)
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('关键词词云')
    plt.tight_layout()
    plt.show()
    print(f"词云已保存: {path}")

In [ ]:
# ========================================
# 1.5 一键分析 (analyze方法)
# ========================================

print("=== 一键NLP分析 ===")
result = analyze_text(sample_text)

print(f"词数: {result.word_count}")
print(f"实体数: {len(result.entities)}")
print(f"关键词数: {len(result.keywords)}")
print(f"摘要长度: {len(result.summary)} 字符")

# 转为字典 (方便保存为JSON)
result_dict = result.to_dict()
print(f"\n结果字典键: {list(result_dict.keys())}")

---
## 总结

### NLPPipeline 提供的方法:
| 方法 | 功能 | 模型 | 首次运行 |
|------|------|------|---------|
| `extract_keywords()` | 关键词提取 | KeyBERT | 下载~400MB |
| `extract_entities()` | 实体识别 | BERT-NER | 下载~400MB |
| `generate_summary()` | 摘要生成 | BART | 下载~1.5GB |
| `generate_wordcloud()` | 词云图 | 无需模型 | 即时可用 |
| `analyze()` | 一键全部分析 | 全部 | 全部下载 |

### 下一步: 运行 `05_knowledge_graph.ipynb` 构建知识图谱